# Silver: Job Standardization

This notebook standardizes raw bronze job data into the silver layer with:
- **Canonicalization**: Normalize company names, job titles, locations
- **Field mapping**: Map source-specific fields to standard schema
- **Data type standardization**: Dates, salaries, employment types
- **Record hashing**: Generate fingerprints for change detection
- **Idempotency**: Safe to re-run on same data

## Architecture

**Input**: `bronze.bronze_job_snapshot`  
**Output**: Staged standardized records ready for CDC detection  
**Mode**: Incremental (process new batches only)

In [0]:
# Databricks notebook parameters
dbutils.widgets.text("batch_id", "", "Batch ID to process (leave empty for latest)")
dbutils.widgets.text("source_filter", "", "Filter by source (comma-separated, empty for all)")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"], "Processing Mode")

# Get parameter values
batch_id = dbutils.widgets.get("batch_id").strip()
source_filter = dbutils.widgets.get("source_filter").strip()
mode = dbutils.widgets.get("mode")

print(f"Parameters:")
print(f"  Batch ID: {batch_id if batch_id else 'Latest'}")
print(f"  Source Filter: {source_filter if source_filter else 'All sources'}")
print(f"  Mode: {mode}")

In [0]:
import hashlib
import json
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Configuration
CATALOG = "workspace"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"
METADATA_SCHEMA = f"{CATALOG}.metadata"

# Current run metadata
run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
run_timestamp = F.current_timestamp()

print(f"Run ID: {run_id}")
print(f"Bronze Schema: {BRONZE_SCHEMA}")
print(f"Silver Schema: {SILVER_SCHEMA}")

In [0]:
# Load bronze snapshot data
query = f"SELECT * FROM {BRONZE_SCHEMA}.bronze_job_snapshot WHERE 1=1"

# Apply batch filter
if batch_id:
    query += f" AND batch_id = '{batch_id}'"
else:
    # Get latest batch if not specified
    latest_batch_df = spark.sql(f"""
        SELECT MAX(batch_id) as latest_batch 
        FROM {BRONZE_SCHEMA}.bronze_job_snapshot
    """)
    latest_batch = latest_batch_df.first()["latest_batch"]
    if latest_batch:
        query += f" AND batch_id = '{latest_batch}'"
        print(f"Processing latest batch: {latest_batch}")

# Apply source filter
if source_filter:
    sources = [f"'{s.strip()}'" for s in source_filter.split(",")]
    query += f" AND source_name IN ({','.join(sources)})"

bronze_df = spark.sql(query)
record_count = bronze_df.count()
print(f"Loaded {record_count} records from bronze")

if record_count == 0:
    dbutils.notebook.exit({"status": "success", "message": "No records to process", "records_processed": 0})

In [0]:
def normalize_company_name(company):
    """Normalize company name for matching"""
    if not company:
        return None
    return company.strip().upper().replace(",", "").replace(".", "").replace("INC", "").replace("LLC", "").replace("LTD", "").replace("GMBH", "").strip()

def normalize_title(title):
    """Normalize job title"""
    if not title:
        return None
    return title.strip().upper()

def normalize_location(location):
    """Normalize location string"""
    if not location:
        return None
    return location.strip().title()

def detect_remote_type(location, remote_flag, description):
    """Detect if job is remote, hybrid, or onsite"""
    if not location and not description:
        return "UNKNOWN"
    
    location_lower = (location or "").lower()
    desc_lower = (description or "").lower()
    
    if remote_flag or "remote" in location_lower or "remote" in desc_lower:
        if "hybrid" in location_lower or "hybrid" in desc_lower:
            return "HYBRID"
        return "REMOTE"
    return "ONSITE"

def generate_record_hash(*fields):
    """Generate deterministic hash for change detection"""
    concatenated = "|".join([str(f) if f is not None else "" for f in fields])
    return hashlib.sha256(concatenated.encode()).hexdigest()

# Register UDFs
normalize_company_udf = F.udf(normalize_company_name, StringType())
normalize_title_udf = F.udf(normalize_title, StringType())
normalize_location_udf = F.udf(normalize_location, StringType())
detect_remote_udf = F.udf(detect_remote_type, StringType())
generate_hash_udf = F.udf(generate_record_hash, StringType())

print("Standardization functions registered")

In [0]:
# Parse JSON payload to struct
parsed_df = bronze_df.withColumn("parsed_payload", F.from_json(F.col("payload_json"), "company_name STRING, title STRING, description STRING, location STRING, remote BOOLEAN, url STRING, created_at LONG, candidate_required_location STRING"))

# Apply source-specific field mapping and standardization
standardized_df = parsed_df.select(
    # Identity
    F.col("source_name"),
    F.col("source_job_id"),
    F.concat_ws("|", F.col("source_name"), F.col("source_job_id")).alias("source_job_key"),
    
    # Company
    F.col("parsed_payload.company_name").alias("company_name_raw"),
    normalize_company_udf(F.col("parsed_payload.company_name")).alias("company_name_norm"),
    
    # Job title
    F.col("parsed_payload.title").alias("title_raw"),
    normalize_title_udf(F.col("parsed_payload.title")).alias("title_normalized"),
    
    # Description
    F.col("parsed_payload.description").alias("description_raw"),
    
    # Location
    F.coalesce(F.col("parsed_payload.location"), F.col("parsed_payload.candidate_required_location")).alias("location_raw"),
    normalize_location_udf(F.coalesce(F.col("parsed_payload.location"), F.col("parsed_payload.candidate_required_location"))).alias("location_norm"),
    
    # Remote type detection
    detect_remote_udf(
        F.coalesce(F.col("parsed_payload.location"), F.col("parsed_payload.candidate_required_location")),
        F.col("parsed_payload.remote"),
        F.col("parsed_payload.description")
    ).alias("remote_type"),
    
    # URL
    F.col("parsed_payload.url").alias("source_url"),
    
    # Timestamps
    F.col("parsed_payload.created_at").cast("timestamp").alias("posted_at"),
    F.col("ingestion_timestamp").alias("last_seen"),
    
    # Batch tracking
    F.col("batch_id"),
    F.lit(run_timestamp).alias("processed_at")
)

# Generate record hash for change detection
standardized_df = standardized_df.withColumn(
    "record_hash",
    generate_hash_udf(
        F.col("company_name_norm"),
        F.col("title_normalized"),
        F.col("description_raw"),
        F.col("location_norm"),
        F.col("remote_type")
    )
)

# Add silver layer metadata
standardized_df = standardized_df.withColumn(
    "is_active", F.lit(True)
).withColumn(
    "soft_delete_flag", F.lit(False)
).withColumn(
    "soft_delete_reason", F.lit(None).cast("string")
)

print(f"Standardization complete: {standardized_df.count()} records")
display(standardized_df.limit(5))

In [0]:
# Create staging table for CDC detection
staging_table = f"{SILVER_SCHEMA}.silver_jobs_staging"

# Create staging table if it doesn't exist
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {staging_table} (
  source_name STRING,
  source_job_id STRING,
  source_job_key STRING,
  company_name_raw STRING,
  company_name_norm STRING,
  title_raw STRING,
  title_normalized STRING,
  description_raw STRING,
  location_raw STRING,
  location_norm STRING,
  remote_type STRING,
  source_url STRING,
  posted_at TIMESTAMP,
  last_seen TIMESTAMP,
  batch_id STRING,
  processed_at TIMESTAMP,
  record_hash STRING,
  is_active BOOLEAN,
  soft_delete_flag BOOLEAN,
  soft_delete_reason STRING
)
USING DELTA
COMMENT 'Staging area for standardized jobs before CDC detection'
""")

# Write standardized data to staging (overwrite for idempotency)
standardized_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(staging_table)

staged_count = spark.table(staging_table).count()
print(f"Staged {staged_count} records to {staging_table}")

In [0]:
# Generate summary statistics
summary_df = spark.sql(f"""
SELECT 
  source_name,
  COUNT(*) as record_count,
  COUNT(DISTINCT company_name_norm) as unique_companies,
  COUNT(DISTINCT title_normalized) as unique_titles,
  COUNT(DISTINCT location_norm) as unique_locations,
  SUM(CASE WHEN remote_type = 'REMOTE' THEN 1 ELSE 0 END) as remote_jobs,
  SUM(CASE WHEN remote_type = 'HYBRID' THEN 1 ELSE 0 END) as hybrid_jobs,
  SUM(CASE WHEN remote_type = 'ONSITE' THEN 1 ELSE 0 END) as onsite_jobs
FROM {staging_table}
GROUP BY source_name
""")

print("\n=== Standardization Summary ===")
display(summary_df)

# Return success
result = {
    "status": "success",
    "run_id": run_id,
    "records_processed": staged_count,
    "staging_table": staging_table,
    "next_step": "Run nb_silver_detect_cdc"
}

print(f"\n{json.dumps(result, indent=2)}")
dbutils.notebook.exit(json.dumps(result))